# Dividend Initiation and Omission — Data Build

Download Compustat Fundamentals Annual, apply sample filters, construct dividend-event targets, engineer 34 predictors, and export ML-ready CSVs.

The pipeline is intentionally limited to data construction. Descriptive statistics, modelling, and evaluation belong in separate notebooks so that re-running the data build does not trigger costly downstream computation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

# Paths
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_CACHE       = DATA_DIR / 'compustat_funda_dividend_ml_raw_1963_2025.csv.gz'
FULL_PANEL_CSV  = DATA_DIR / 'dividend_full_engineered_panel_1963_2025.csv'
INITIATION_CSV  = DATA_DIR / 'ml_dividend_initiations_1990_2025.csv'
OMISSION_CSV    = DATA_DIR / 'ml_dividend_omissions_1990_2025.csv'
FEATURES_CSV    = DATA_DIR / 'ml_feature_list.csv'

# Set True when you want to force a fresh WRDS download even if RAW_CACHE exists.
FORCE_DOWNLOAD = False

START_ANALYSIS_YEAR      = 1990
END_ANALYSIS_YEAR        = 2025
MIN_MICHAELY_LISTING_AGE = 2

# Positive accounting-base screen. Compustat fundamentals are in millions;
# the thesis main sample uses strictly positive accounting values rather than
# the stricter $500k/$250k Fama-French dollar thresholds.
MIN_ASSETS = 0
MIN_EQUITY = 0

## 1. Download Compustat Data

The extraction starts in 1963 — three decades before the 1990 ML estimation window — to eliminate left-censoring when classifying first-time dividend payers. Without pre-sample history, firms that paid dividends in the 1970s–1980s would incorrectly appear as initiators in the 1990s.

In [2]:
query = """
SELECT
       -- Identifiers
       gvkey,
       datadate,
       fyear,
       tic,
       cusip,
       conm,
       fic,
       curcd,
       cshtr_f,

       -- Dividend & payout variables: target construction / payout channels
       dvc,
       dvt,
       dvpsx_f,
       prstkc,

       -- Income statement & profitability
       sale,
       revt,
       ni,
       ib,
       oibdp,
       oiadp,
       ebit,
       dp,
       pi,
       spi,
       txt,
       txdi,
       txpd,
       xint,
       xrd,

       -- Balance sheet: assets
       at,
       act,
       che,
       ppent,
       gdwl,

       -- Balance sheet: liabilities and equity
       lt,
       lct,
       dltt,
       dlc,
       ceq,
       seq,
       re,
       pstkrv,
       pstkl,
       pstk,
       txditc,

       -- Cash flow statement
       oancf,
       capx,
       fincf,

       -- Market, shares, and labour
       csho,
       prcc_f,
       emp,
       xlr

FROM comp.funda
WHERE indfmt = 'INDL'
  AND datafmt = 'STD'
  AND popsrc = 'D'
  AND consol = 'C'
  AND fic = 'USA'
  AND curcd = 'USD'
  AND datadate BETWEEN '1963-01-01' AND '2025-12-31'
ORDER BY gvkey, datadate;
"""

expected_raw_columns = [
    'gvkey', 'datadate', 'fyear', 'tic', 'cusip', 'conm', 'fic', 'curcd', 'cshtr_f',
    'dvc', 'dvt', 'dvpsx_f', 'prstkc',
    'sale', 'revt', 'ni', 'ib', 'oibdp', 'oiadp', 'ebit', 'dp', 'pi', 'spi',
    'txt', 'txdi', 'txpd', 'xint', 'xrd',
    'at', 'act', 'che', 'ppent', 'gdwl',
    'lt', 'lct', 'dltt', 'dlc', 'ceq', 'seq', 're', 'pstkrv', 'pstkl', 'pstk', 'txditc',
    'oancf', 'capx', 'fincf',
    'csho', 'prcc_f', 'emp', 'xlr',
]

if RAW_CACHE.exists() and not FORCE_DOWNLOAD:
    raw = pd.read_csv(RAW_CACHE, parse_dates=['datadate'], low_memory=False)
    print(f'Loaded cached raw Compustat data: {RAW_CACHE}')
else:
    import wrds

    conn = wrds.Connection()
    raw = conn.raw_sql(query, date_cols=['datadate'])
    raw.to_csv(RAW_CACHE, index=False)
    print(f'Downloaded and cached raw Compustat data: {RAW_CACHE}')

# Keep only variables used by the thesis specification. This makes old caches
# harmless if they contain columns from previous notebook versions.
raw = raw.loc[:, [c for c in expected_raw_columns if c in raw.columns]].copy()

print(f'Raw shape: {raw.shape}')
print(f'Date range: {raw["datadate"].min()} -> {raw["datadate"].max()}')
print(f'Unique gvkeys: {raw["gvkey"].nunique():,}')
raw.head()


Loaded cached raw Compustat data: /Users/matyaslezatka/Documents/Thesis/Cleaned/data/compustat_funda_dividend_ml_raw_1963_2025.csv.gz
Raw shape: (469590, 51)
Date range: 1963-01-31 00:00:00 -> 2025-12-31 00:00:00
Unique gvkeys: 36,856


,gvkey,datadate,fyear,tic,cusip,conm,fic,curcd,cshtr_f,dvc,dvt,dvpsx_f,prstkc,sale,revt,ni,ib,oibdp,oiadp,ebit,dp,pi,spi,txt,txdi,txpd,xint,xrd,at,act,che,ppent,gdwl,lt,lct,dltt,dlc,ceq,seq,re,pstkrv,pstkl,pstk,txditc,oancf,capx,fincf,csho,prcc_f,emp,xlr
0,1000,1963-12-31,1963,AE.2,000032102,A & E PLASTIK PAK INC,USA,USD,NaN,0.0,0.0,0.0,NaN,1.457,1.457,0.003,0.003,0.046,0.000,0.000,0.046,0.004,0.0,0.001,NaN,NaN,0.020,NaN,NaN,0.408,NaN,0.431,NaN,0.345,0.322,0.015,NaN,0.553,0.553,0.052,0.0,0.0,0.0,0.008,NaN,NaN,NaN,0.186,NaN,NaN,NaN
1,1000,1964-12-31,1964,AE.2,000032102,A & E PLASTIK PAK INC,USA,USD,NaN,0.0,0.0,0.0,NaN,2.032,2.032,0.052,0.039,0.127,0.074,0.074,0.053,0.059,0.0,0.020,NaN,NaN,0.033,NaN,1.416,0.718,0.269,0.563,NaN,0.809,0.267,0.522,0.088,0.607,0.607,0.001,0.0,0.0,0.0,0.020,NaN,NaN,NaN,0.196,NaN,NaN,NaN
2,1000,1965-12-31,1965,AE.2,000032102,A & E PLASTIK PAK INC,USA,USD,NaN,0.0,0.0,0.0,NaN,1.688,1.688,-0.197,-0.197,-0.160,-0.242,-0.242,0.082,-0.300,0.0,-0.103,NaN,NaN,0.062,NaN,2.310,0.725,0.031,1.397,NaN,1.818,0.623,1.154,0.300,0.491,0.491,-0.196,0.0,0.0,0.0,0.000,NaN,NaN,NaN,0.206,NaN,NaN,NaN
3,1000,1966-12-31,1966,AE.2,000032102,A & E PLASTIK PAK INC,USA,USD,NaN,0.0,0.0,0.0,NaN,4.032,4.032,0.164,0.201,0.525,0.350,0.350,0.175,0.201,0.0,0.000,0.0,NaN,0.095,NaN,2.430,1.015,0.063,1.338,NaN,1.596,0.446,1.109,0.124,0.834,0.834,-0.031,0.0,0.0,0.0,0.000,NaN,0.118,NaN,0.219,NaN,NaN,NaN
4,1000,1967-12-31,1967,AE.2,000032102,A & E PLASTIK PAK INC,USA,USD,NaN,0.0,0.0,0.0,NaN,3.594,3.594,-0.090,-0.090,0.201,0.003,0.003,0.198,-0.090,0.0,0.000,0.0,NaN,0.097,NaN,2.456,1.004,0.029,1.325,NaN,1.712,0.376,1.295,0.086,0.744,0.744,-0.121,0.0,0.0,0.0,0.000,NaN,0.180,NaN,0.219,NaN,NaN,NaN


## 2. Rename, Type-Clean, and De-Duplicate

Map raw Compustat mnemonics to readable names, coerce numeric columns, and resolve the small number of duplicate `FirmID × FiscalYear` records by keeping the latest `FiscalDate` entry. Deduplication prevents one-to-many joins and inflated event counts downstream.

In [3]:
rename_dict = {
    # identifiers
    'gvkey': 'FirmID',
    'datadate': 'FiscalDate',
    'fyear': 'FiscalYear',
    'tic': 'Ticker',
    'cusip': 'CUSIP',
    'conm': 'CompanyName',
    'fic': 'FIC',
    'curcd': 'Currency',

    # dividends / payout
    'dvc': 'DivCash',
    'dvt': 'DivTotal',
    'dvpsx_f': 'DivPS_Fiscal',
    'prstkc': 'Repurchases',

    # income statement / profitability
    'sale': 'Sales',
    'revt': 'RevenueTotal',
    'ni': 'NetIncome',
    'ib': 'IncomeBeforeExtra',
    'oibdp': 'OperatingIncomeBeforeDepreciation',
    'oiadp': 'OperatingIncomeAfterDepreciation',
    'ebit': 'EBIT',
    'dp': 'Depreciation',
    'pi': 'PretaxIncome',
    'spi': 'SpecialItems',
    'txt': 'IncomeTaxExpense',
    'txdi': 'DeferredTaxesIncomeStatement',
    'txpd': 'CashTaxesPaid',
    'xint': 'InterestExpense',
    'xrd': 'RD_Expense',

    # balance sheet assets
    'at': 'TotalAssets',
    'act': 'CurrentAssets',
    'che': 'CashAndEquiv',
    'ppent': 'PPE_Net',
    'gdwl': 'Goodwill',

    # balance sheet liabilities/equity
    'lt': 'TotalLiabilities',
    'lct': 'CurrentLiabilities',
    'dltt': 'LongTermDebt',
    'dlc': 'DebtCurrent',
    'ceq': 'CommonEquity',
    'seq': 'StockholdersEquity',
    're': 'RetainedEarnings',
    'pstkrv': 'PreferredStockRedemption',
    'pstkl': 'PreferredStockLiquidating',
    'pstk': 'PreferredStock',
    'txditc': 'DeferredTaxesInvestmentCredit',

    # cash flow statement
    'oancf': 'OpCashFlow',
    'capx': 'CapEx',
    'fincf': 'FinancingCashFlow',

    # market, shares, labour
    'csho': 'SharesOut',
    'prcc_f': 'StockPriceFiscalClose',
    'emp': 'Employees',
    'xlr': 'StaffExpense',
    'cshtr_f': 'SharesTradedFiscal',
}

df = raw.rename(columns=rename_dict).copy()

# Dates and years
df['FiscalDate'] = pd.to_datetime(df['FiscalDate'], errors='coerce')
df['FiscalYear'] = pd.to_numeric(df['FiscalYear'], errors='coerce').astype('Int64')

id_cols = ['FirmID', 'Ticker', 'CUSIP', 'CompanyName', 'FIC', 'Currency']
for col in id_cols:
    if col in df.columns:
        df[col] = df[col].astype('string')

numeric_cols = [c for c in df.columns if c not in id_cols + ['FiscalDate']]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Keep one record per FirmID-FiscalYear. If duplicates exist, keep the latest FiscalDate.
df = df.sort_values(['FirmID', 'FiscalYear', 'FiscalDate']).reset_index(drop=True)
duplicates_before = int(df.duplicated(['FirmID', 'FiscalYear']).sum())
df = df.drop_duplicates(['FirmID', 'FiscalYear'], keep='last').reset_index(drop=True)

print(f'Duplicated FirmID-FiscalYear rows removed: {duplicates_before:,}')
print(f'Panel shape after type cleaning: {df.shape}')
print(f'Fiscal-year range: {df["FiscalYear"].min()} -> {df["FiscalYear"].max()}')


Duplicated FirmID-FiscalYear rows removed: 11
Panel shape after type cleaning: (469579, 51)
Fiscal-year range: 1962 -> 2025


## 3. Compustat INDL Universe and Full-History Age

Listing age is computed *before* accounting filters so that early weak observations do not reset a firm's Compustat tenure. A firm first recorded in 1970 enters the 1990 estimation window with `Listing_Age = 20`, preserving the maturity signal that life-cycle theory relies on (DeAngelo, DeAngelo & Stulz, 2006).

No market-cap screen is applied at this stage: missing `prcc_f` or `csho` should make market-based predictors missing, not remove otherwise valid accounting observations from the at-risk universe.

In [4]:
pre_universe_rows = len(df)

df = df.copy()

# Listing age is computed before accounting filters so early weak/missing
# accounting observations do not reset the firm's observed Compustat age.
df = df.sort_values(['FirmID', 'FiscalYear', 'FiscalDate']).reset_index(drop=True)
df['first_compustat_year_full'] = df.groupby('FirmID')['FiscalYear'].transform('min')
df['listing_age_full'] = df['FiscalYear'] - df['first_compustat_year_full']

df['first_compustat_year'] = df['first_compustat_year_full']
df['listing_age'] = df['listing_age_full']

print('Compustat industrial-format universe diagnostics:')
print(f'Rows before universe step: {pre_universe_rows:,}')
print(f'Rows after universe step:  {len(df):,}')
print(f'Rows removed by industry-code filters: 0')
print(f'Unique firms after universe step: {df["FirmID"].nunique():,}')


Compustat industrial-format universe diagnostics:
Rows before universe step: 469,579
Rows after universe step:  469,579
Rows removed by industry-code filters: 0
Unique firms after universe step: 36,856


## 4. Dividend Payer Status, Accounting Base, and Event Targets

Dividend history is built *before* the positive accounting-base filter. Applying accounting screens first would erase earlier dividend payments for firms with temporarily weak fundamentals, creating false first-time initiations.

Payer status uses `DivPS > 0 OR DivCash > 0` because Compustat has a material number of observations with positive common dividends (`dvc`) but zero or missing per-share data. Using per-share data alone would misclassify these as omissions.

**Initiation** (Michaely et al., 1995): first cash dividend payment after ≥ 2 years of Compustat coverage, from a firm with no prior observed dividend. Re-initiations are excluded via the cumulative-max `PaidPriorEver` flag.

**Omission** (Michaely et al., 1995, criterion 3): zero dividends in year *t* having paid in both *t − 1* and *t − 2*, with consecutive Compustat coverage across all three years.

In [5]:
df['DivPS_Fiscal'] = pd.to_numeric(df['DivPS_Fiscal'], errors='coerce')
df['DivCash'] = pd.to_numeric(df['DivCash'], errors='coerce')
df['SharesOut'] = pd.to_numeric(df['SharesOut'], errors='coerce')

fallback_divps = df['DivCash'] / df['SharesOut'].replace(0, np.nan)

df['DivPS'] = (
    df['DivPS_Fiscal']
      .fillna(fallback_divps)
      .fillna(0)
      .clip(lower=0)
)

payer_mismatch = pd.Series({
    'DivPS_positive_total': int(df['DivPS'].gt(0).sum()),
    'DivCash_positive_DivPS_zero': int((df['DivCash'].fillna(0).gt(0) & df['DivPS'].eq(0)).sum()),
    'DivPS_positive_DivCash_missing_or_zero': int((df['DivPS'].gt(0) & ~df['DivCash'].fillna(0).gt(0)).sum()),
})

print('Dividend payer-definition diagnostics:')
print(payer_mismatch.to_string())
print()

# Missing dvc is a non-payer for the Baker-Wurgler-style payer split.
df['DividendPayer_DVC'] = df['DivCash'].fillna(0).gt(0)

# Payer status uses cleaned per-share dividends plus positive total common
# dividends. This avoids false omissions when DivPS is unavailable but dvc > 0.
df['DividendPaid'] = df['DivPS'].gt(0) | df['DivCash'].fillna(0).gt(0)
df['DividendPaidInt'] = df['DividendPaid'].astype('int8')

df['FiscalYear_lag'] = df.groupby('FirmID')['FiscalYear'].shift(1)
df['has_t_minus_1'] = (df['FiscalYear'] - df['FiscalYear_lag']).eq(1).fillna(False)

df['FiscalYear_lag2'] = df.groupby('FirmID')['FiscalYear'].shift(2)
df['has_t_minus_2'] = (df['FiscalYear_lag'] - df['FiscalYear_lag2']).eq(1).fillna(False)

df['DividendPaid_lag1'] = (
    df.groupby('FirmID')['DividendPaid']
      .shift(1)
      .astype('boolean')
      .fillna(False)
      .astype(bool)
)
df['DividendPaid_lag2'] = (
    df.groupby('FirmID')['DividendPaid']
      .shift(2)
      .astype('boolean')
      .fillna(False)
      .astype(bool)
)
df['DividendPaid_lag'] = df['DividendPaid_lag1']

df['PaidPriorEver'] = (
    df.groupby('FirmID')['DividendPaidInt']
      .transform(lambda s: s.shift(1).fillna(0).cummax())
      .astype(bool)
)

pre_accounting_rows = len(df)
positive_fundamentals = (
    df['TotalAssets'].gt(MIN_ASSETS) &
    df['CommonEquity'].gt(MIN_EQUITY) &
    df['Sales'].gt(0)
)

accounting_diagnostics = pd.Series({
    'rows_before_positive_accounting_filter': int(pre_accounting_rows),
    'non_positive_or_missing_assets': int((~df['TotalAssets'].gt(MIN_ASSETS)).sum()),
    'non_positive_or_missing_common_equity': int((~df['CommonEquity'].gt(MIN_EQUITY)).sum()),
    'non_positive_or_missing_sales': int((~df['Sales'].gt(0)).sum()),
})

df = df.loc[positive_fundamentals].copy().reset_index(drop=True)
df['first_compustat_year'] = df['first_compustat_year_full']
df['listing_age'] = df['listing_age_full']

# Model-predictor lags are computed after the accounting-base filter. These
# flags keep event-history coverage separate from valid t-1 predictor coverage.
df = df.sort_values(['FirmID', 'FiscalYear', 'FiscalDate']).reset_index(drop=True)
df['FiscalYear_lag_model'] = df.groupby('FirmID')['FiscalYear'].shift(1)
df['has_model_lag1'] = df['FiscalYear'].sub(df['FiscalYear_lag_model']).eq(1).fillna(False)

df['FiscalYear_lag2_model'] = df.groupby('FirmID')['FiscalYear'].shift(2)
df['has_model_lag2'] = df['FiscalYear_lag_model'].sub(df['FiscalYear_lag2_model']).eq(1).fillna(False)

candidate_init = (
    df['DividendPaid'] &
    (~df['PaidPriorEver']) &
    df['has_t_minus_1'] &
    (df['listing_age'] >= MIN_MICHAELY_LISTING_AGE)
)

df['target_initiation'] = candidate_init.astype('int8')

first_init_year = (
    df.loc[df['target_initiation'].eq(1)]
      .groupby('FirmID')['FiscalYear']
      .min()
      .rename('first_init_year')
)
df = df.join(first_init_year, on='FirmID')
df['target_initiation'] = (
    df['target_initiation'].eq(1) & df['FiscalYear'].eq(df['first_init_year'])
).astype('int8')
df = df.drop(columns=['first_init_year'])

df['target_omission'] = (
    (~df['DividendPaid']) &
    df['DividendPaid_lag1'] &
    df['DividendPaid_lag2'] &
    df['has_t_minus_1'] &
    df['has_t_minus_2']
).astype('int8')

print('Positive accounting-base diagnostics:')
print(accounting_diagnostics.to_string())
print()
print(f'Rows after positive accounting-base filter: {len(df):,}')
print(f'Rows removed by accounting filter: {pre_accounting_rows - len(df):,}')
print(f'Unique firms after accounting filter: {df["FirmID"].nunique():,}')
print(f'Rows without valid model t-1 after accounting filter: {int((~df["has_model_lag1"]).sum()):,}')
print()
print(f'Initiation events in full filtered panel: {int(df["target_initiation"].sum()):,}')
print(f'Unique initiating firms: {df.loc[df["target_initiation"].eq(1), "FirmID"].nunique():,}')
print(f'Omission events in full filtered panel: {int(df["target_omission"].sum()):,}')
print(f'Unique omitting firms: {df.loc[df["target_omission"].eq(1), "FirmID"].nunique():,}')


Dividend payer-definition diagnostics:
DivPS_positive_total                      199019
DivCash_positive_DivPS_zero                12979
DivPS_positive_DivCash_missing_or_zero     55166

Positive accounting-base diagnostics:
rows_before_positive_accounting_filter    469579
non_positive_or_missing_assets             83722
non_positive_or_missing_common_equity     126663
non_positive_or_missing_sales              98024

Rows after positive accounting-base filter: 332,358
Rows removed by accounting filter: 137,221
Unique firms after accounting filter: 26,682
Rows without valid model t-1 after accounting filter: 32,243

Initiation events in full filtered panel: 4,737
Unique initiating firms: 4,737
Omission events in full filtered panel: 4,925
Unique omitting firms: 4,145


## 5. Helper Functions

Reusable arithmetic primitives for ratio construction. `safe_div` returns `NaN` when the denominator is missing or zero, preventing silent `inf` values from propagating into features.

In [6]:
def safe_div(num, den):
    """Vectorized division returning NaN when denominator is missing or zero."""
    return num / den.replace(0, np.nan)


def safe_log(series):
    """Natural log only where the input is strictly positive."""
    return np.log(series.where(series > 0))


def weighted_average(group, value_col, weight_col):
    valid = group[value_col].notna() & group[weight_col].notna() & group[weight_col].gt(0)
    if not valid.any():
        return np.nan
    values = group.loc[valid, value_col]
    weights = group.loc[valid, weight_col]
    return np.average(values, weights=weights)

## 6. Firm-Level Features

Thirty-three firm-level predictors spanning eight theoretical blocks (life-cycle, growth options, profitability, agency, risk, taxation, transaction costs). Features are computed contemporaneously here; lagging to `t − 1` happens in §8.

Items that are commonly missing because the activity does not apply (R&D, goodwill, repurchases, short-term debt) are zero-imputed *before* ratio construction. This reflects the economic reality that unreported R&D expense is zero R&D, not unknown R&D.

In [7]:
# Zero-impute items that are commonly missing because they are not reported/applicable.
df['RD_Expense_filled'] = df['RD_Expense'].fillna(0)
df['Goodwill_filled'] = df['Goodwill'].fillna(0)
df['Repurchases_filled'] = df['Repurchases'].fillna(0)
df['DebtCurrent_filled'] = df['DebtCurrent'].fillna(0)
df['LongTermDebt_filled'] = df['LongTermDebt'].fillna(0)

AT = df['TotalAssets']
CEQ = df['CommonEquity']
SALE = df['Sales']
MCAP = df['StockPriceFiscalClose'] * df['SharesOut']
df['MarketCap'] = MCAP.where(MCAP > 0)

# Life-cycle and maturity
df['RE_TE'] = safe_div(df['RetainedEarnings'], CEQ)
df['RE_TA'] = safe_div(df['RetainedEarnings'], AT)
df['TE_TA'] = safe_div(CEQ, AT)
df['Size'] = safe_log(AT)
df['Log_MarketCap'] = safe_log(df['MarketCap'])
df['Sales_Growth'] = (
    df.groupby('FirmID')['Sales']
      .pct_change()
      .replace([np.inf, -np.inf], np.nan)
      .where(df['has_model_lag1'])
)
df['Asset_Growth'] = (
    df.groupby('FirmID')['TotalAssets']
      .pct_change()
      .replace([np.inf, -np.inf], np.nan)
      .where(df['has_model_lag1'])
)
df['Listing_Age'] = df['listing_age']

# Growth options and investment opportunities
df['Market_to_Book'] = safe_div(AT - CEQ + df['MarketCap'], AT)
df['CapEx_to_Assets'] = safe_div(df['CapEx'], AT)
df['RD_assets'] = safe_div(df['RD_Expense_filled'], AT)
df['Price_to_Sales'] = safe_div(df['MarketCap'], SALE)

# Signaling and profitability
df['ROA'] = safe_div(df['NetIncome'], AT)
df['ROE'] = safe_div(df['NetIncome'], CEQ)
df['OpCF_to_Assets'] = safe_div(df['OpCashFlow'], AT)
df['ORA'] = safe_div(df['OperatingIncomeBeforeDepreciation'], AT)
df['Profit_Margin'] = safe_div(df['IncomeBeforeExtra'], SALE)
df['Accruals'] = safe_div(df['IncomeBeforeExtra'] - df['OpCashFlow'], AT)

# Free cash flow / agency
df['Cash_to_Assets'] = safe_div(df['CashAndEquiv'], AT)
df['FCF_to_Assets'] = safe_div(df['OpCashFlow'] - df['CapEx'], AT)
df['Repurchase_to_Assets'] = safe_div(df['Repurchases_filled'], AT)
df['Goodwill_to_Assets'] = safe_div(df['Goodwill_filled'], AT)

# Conservatism and risk
df['Leverage'] = safe_div(df['LongTermDebt_filled'] + df['DebtCurrent_filled'], AT)
df['LTDA'] = safe_div(df['LongTermDebt_filled'], AT)
df['Current_Ratio'] = safe_div(df['CurrentAssets'], df['CurrentLiabilities'])

roa_lag1 = df.groupby('FirmID')['ROA'].shift(1)
roa_lag2 = df.groupby('FirmID')['ROA'].shift(2)
two_year_vol = pd.concat(
    [roa_lag1.rename('ROA_lag1'), df['ROA'].rename('ROA')],
    axis=1,
).std(axis=1, ddof=1)
three_year_vol = pd.concat(
    [roa_lag2.rename('ROA_lag2'), roa_lag1.rename('ROA_lag1'), df['ROA'].rename('ROA')],
    axis=1,
).std(axis=1, ddof=1)

df['Earnings_Volatility'] = np.nan
df.loc[df['has_model_lag1'], 'Earnings_Volatility'] = two_year_vol[df['has_model_lag1']]
df.loc[
    df['has_model_lag1'] & df['has_model_lag2'],
    'Earnings_Volatility',
] = three_year_vol[df['has_model_lag1'] & df['has_model_lag2']]

df['Interest_Coverage'] = safe_div(df['EBIT'], df['InterestExpense'])
df['FAT'] = safe_div(df['DebtCurrent_filled'] + 0.5 * df['LongTermDebt_filled'], AT)
df['LCTAT'] = safe_div(df['CurrentLiabilities'], AT)
df['Labor_Intensity'] = safe_div(df['Employees'], df['PPE_Net'])

# Corporate taxation
df['GAAP_ETR'] = safe_div(df['IncomeTaxExpense'], df['PretaxIncome'])
df['Cash_ETR'] = safe_div(df['CashTaxesPaid'], df['PretaxIncome'] - df['SpecialItems'].fillna(0))

# Transaction cost theory
df['Share_Turnover'] = safe_div(df['SharesTradedFiscal'], df['SharesOut'])

base_features = [
    'RE_TE', 'RE_TA', 'TE_TA', 'Size', 'Log_MarketCap', 'Sales_Growth', 'Asset_Growth', 'Listing_Age',
    'Market_to_Book', 'CapEx_to_Assets', 'RD_assets', 'Price_to_Sales',
    'ROA', 'ROE', 'OpCF_to_Assets', 'ORA', 'Profit_Margin', 'Accruals',
    'Cash_to_Assets', 'FCF_to_Assets', 'Repurchase_to_Assets', 'Goodwill_to_Assets',
    'Leverage', 'LTDA', 'Current_Ratio', 'Earnings_Volatility', 'Interest_Coverage', 'FAT', 'LCTAT', 'Labor_Intensity',
    'GAAP_ETR', 'Cash_ETR', 'Share_Turnover',
]

# Replace infinite values created by extreme denominators with missing values.
df[base_features] = df[base_features].replace([np.inf, -np.inf], np.nan)

print(f'Base features from variables.pdf: {len(base_features)}')
print(pd.Series(base_features).to_string(index=False))


Base features from variables.pdf: 33
               RE_TE
               RE_TA
               TE_TA
                Size
       Log_MarketCap
        Sales_Growth
        Asset_Growth
         Listing_Age
      Market_to_Book
     CapEx_to_Assets
           RD_assets
      Price_to_Sales
                 ROA
                 ROE
      OpCF_to_Assets
                 ORA
       Profit_Margin
            Accruals
      Cash_to_Assets
       FCF_to_Assets
Repurchase_to_Assets
  Goodwill_to_Assets
            Leverage
                LTDA
       Current_Ratio
 Earnings_Volatility
   Interest_Coverage
                 FAT
               LCTAT
     Labor_Intensity
            GAAP_ETR
            Cash_ETR
      Share_Turnover


## 7. Dividend Premium (`DIVPREM`)

Baker & Wurgler (2004) catering theory posits that managers initiate dividends when the market prices payers at a premium over non-payers. The dividend premium is computed annually as:

`log(asset-weighted M/B of payers) − log(asset-weighted M/B of non-payers)`

Payer status uses `dvc > 0` (common cash dividends); missing `dvc` is treated as non-payment. Unlike the 33 firm-level features, `DIVPREM` is a single market-wide scalar per fiscal year.

In [8]:
# Valid market-to-book observations for annual weighted averages.
divprem_source = df[
    df['FiscalYear'].notna() &
    df['Market_to_Book'].notna() &
    df['Market_to_Book'].gt(0) &
    df['TotalAssets'].notna() &
    df['TotalAssets'].gt(0)
].copy()

yearly_mb = (
    divprem_source
    .groupby(['FiscalYear', 'DividendPayer_DVC'])
    .apply(lambda g: weighted_average(g, 'Market_to_Book', 'TotalAssets'))
    .rename('weighted_mb')
    .reset_index()
)

yearly_mb_wide = (
    yearly_mb
    .pivot(index='FiscalYear', columns='DividendPayer_DVC', values='weighted_mb')
    .rename(columns={False: 'weighted_mb_nonpayers', True: 'weighted_mb_payers'})
    .reset_index()
)

yearly_mb_wide['DIVPREM'] = (
    np.log(yearly_mb_wide['weighted_mb_payers'].where(yearly_mb_wide['weighted_mb_payers'] > 0)) -
    np.log(yearly_mb_wide['weighted_mb_nonpayers'].where(yearly_mb_wide['weighted_mb_nonpayers'] > 0))
)

df = df.merge(
    yearly_mb_wide[['FiscalYear', 'weighted_mb_payers', 'weighted_mb_nonpayers', 'DIVPREM']],
    on='FiscalYear',
    how='left',
    validate='many_to_one',
)

print(yearly_mb_wide[['FiscalYear', 'weighted_mb_payers', 'weighted_mb_nonpayers', 'DIVPREM']].tail(10))

DividendPayer_DVC  FiscalYear  weighted_mb_payers  weighted_mb_nonpayers   DIVPREM
54                       2016            1.322459               1.730424 -0.268873
55                       2017            1.368084               1.924029 -0.341010
56                       2018            1.315384               1.928027 -0.382368
57                       2019            1.368681               2.070289 -0.413841
58                       2020            1.397623               2.495708 -0.579799
59                       2021            1.462759               2.574447 -0.565310
60                       2022            1.418820               1.802159 -0.239160
61                       2023            1.465353               2.231352 -0.420511
62                       2024            1.634631               2.390468 -0.380072
63                       2025            1.602323               2.386872 -0.398529


/var/folders/d9/t3jnw_n57470jpcr3_r2zf9m0000gn/T/ipykernel_7101/1574174877.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: weighted_average(g, 'Market_to_Book', 'TotalAssets'))


## 8. Lag Predictors for Forecasting

All firm-level predictors are lagged one fiscal year within firm. The target at year *t* is predicted using features from year *t − 1*, ensuring strict temporal precedence.

Rows without a valid consecutive `t − 1` observation have firm-level lagged predictors set to `NaN` rather than inheriting stale values from `t − k`. This prevents non-consecutive history from contaminating fold-level imputation.

`DIVPREM_lag1` is lagged by calendar year (not by firm) because the dividend premium is market-wide.

In [9]:
firm_features = base_features
context_features = ['DIVPREM']
all_features = firm_features + context_features

# Ensure no accidental duplicated names.
assert len(all_features) == len(set(all_features)), 'Feature list contains duplicates.'

firm_lagged_features = [f'{col}_lag1' for col in firm_features]
context_lagged_features = ['DIVPREM_lag1']

# Firm-level lag: use the previous surviving post-filter observation only when
# it is exactly fiscal year t-1. Otherwise leave the predictor missing.
df[firm_lagged_features] = df.groupby('FirmID')[firm_features].shift(1)
df[firm_lagged_features] = df[firm_lagged_features].replace([np.inf, -np.inf], np.nan)
df.loc[~df['has_model_lag1'], firm_lagged_features] = np.nan

# Market-wide annual lag: prior fiscal-year DIVPREM, same value for all firms.
divprem_lag = (
    yearly_mb_wide[['FiscalYear', 'DIVPREM']]
      .assign(FiscalYear=lambda x: x['FiscalYear'] + 1)
      .rename(columns={'DIVPREM': 'DIVPREM_lag1'})
)

df = df.drop(columns=context_lagged_features, errors='ignore')
df = df.merge(divprem_lag, on='FiscalYear', how='left', validate='many_to_one')
df[context_lagged_features] = df[context_lagged_features].replace([np.inf, -np.inf], np.nan)

lagged_features = firm_lagged_features + context_lagged_features

feature_table = pd.DataFrame({
    'feature_current_year': all_features,
    'feature_model_column': lagged_features,
})
feature_table.to_csv(FEATURES_CSV, index=False)

print(f'Total model features: {len(lagged_features)}')
print(f'Firm-level lagged features: {len(firm_lagged_features)}')
print(f'Market-wide lagged features: {len(context_lagged_features)}')
print(f'Feature list exported to: {FEATURES_CSV}')
feature_table


Total model features: 34
Firm-level lagged features: 33
Market-wide lagged features: 1
Feature list exported to: /Users/matyaslezatka/Documents/Thesis/Cleaned/data/ml_feature_list.csv


,feature_current_year,feature_model_column
0,RE_TE,RE_TE_lag1
1,RE_TA,RE_TA_lag1
2,TE_TA,TE_TA_lag1
3,Size,Size_lag1
4,Log_MarketCap,Log_MarketCap_lag1
5,Sales_Growth,Sales_Growth_lag1
6,Asset_Growth,Asset_Growth_lag1
7,Listing_Age,Listing_Age_lag1
8,Market_to_Book,Market_to_Book_lag1
9,CapEx_to_Assets,CapEx_to_Assets_lag1


## 9. Build ML Risk Sets

Two non-overlapping risk sets define the prediction tasks:

- **Initiation:** never-before-payers (`PaidPriorEver = False`). Listing-age and consecutive-coverage requirements are event-qualification criteria only — they gate which firms can *initiate*, not which firms enter the control group. Non-consecutive non-event rows retain `NaN` lag features and are median-imputed inside each CV fold.

- **Omission:** firms with two consecutive prior annual payments (Michaely et al., 1995, criterion 3). Controls are continuing payers satisfying the same coverage requirement who do not stop.

In [10]:
common_id_cols = [
    'FirmID', 'FiscalDate', 'FiscalYear', 'Ticker', 'CUSIP', 'CompanyName',
    'FIC', 'Currency',
    'first_compustat_year', 'listing_age',
    'has_t_minus_1', 'has_t_minus_2', 'has_model_lag1', 'has_model_lag2',
    'DividendPaid', 'DividendPaid_lag', 'DividendPaid_lag1', 'DividendPaid_lag2', 'PaidPriorEver',
    'DivCash', 'DivPS', 'MarketCap',
]

analysis_year_mask = df['FiscalYear'].between(START_ANALYSIS_YEAR, END_ANALYSIS_YEAR)

# At-risk universe = all never-before-payers (PaidPriorEver = False).
# listing_age >= 2 and has_t_minus_1 are event-qualification criteria only
# (enforced inside candidate_init in Section 4, not here).
# Non-consecutive firm-years are safe because their lag1 features were nulled
# in Section 8 and will be median-imputed inside each CV fold.
initiation_risk_mask = (
    analysis_year_mask &
    (~df['PaidPriorEver']) &
    ((~df['DividendPaid']) | df['target_initiation'].eq(1))
)

# Michaely et al. (1995) criterion 3 (annual): paid in t-1 AND t-2,
# consecutive coverage across t-2, t-1, t.
omission_risk_mask = (
    analysis_year_mask &
    df['has_t_minus_1'] &
    df['has_t_minus_2'] &
    df['DividendPaid_lag1'] &
    df['DividendPaid_lag2']
)

initiation_ml = df.loc[
    initiation_risk_mask,
    common_id_cols + ['target_initiation'] + lagged_features,
].copy()

omission_ml = df.loc[
    omission_risk_mask,
    common_id_cols + ['target_omission'] + lagged_features,
].copy()

print('Target QA checks')
print('-' * 60)

assert (~initiation_ml.loc[
    initiation_ml['target_initiation'].eq(1), 'PaidPriorEver'
]).all()

assert initiation_ml.loc[
    initiation_ml['target_initiation'].eq(1), 'DividendPaid'
].all()

assert omission_ml.loc[
    omission_ml['target_omission'].eq(1), 'DividendPaid_lag1'
].all()

assert omission_ml.loc[
    omission_ml['target_omission'].eq(1), 'DividendPaid_lag2'
].all()

assert (~omission_ml.loc[
    omission_ml['target_omission'].eq(1), 'DividendPaid'
]).all()

assert omission_ml.loc[
    omission_ml['target_omission'].eq(1), 'has_t_minus_1'
].all()

assert omission_ml.loc[
    omission_ml['target_omission'].eq(1), 'has_t_minus_2'
].all()

print('QA passed.')
print()

# Diagnostic summary.
init_events     = int(initiation_ml['target_initiation'].sum())
init_non_events = int((initiation_ml['target_initiation'] == 0).sum())
omit_events     = int(omission_ml['target_omission'].sum())
omit_non_events = int((omission_ml['target_omission'] == 0).sum())

print('Initiation ML sample')
print(f'  rows: {len(initiation_ml):,}')
print(f'  events: {init_events:,}')
print(f'  non-events: {init_non_events:,}')
print(f'  imbalance non-event/event: {init_non_events / init_events:.2f}:1' if init_events else '  no initiation events')
print(f'  year range: {initiation_ml["FiscalYear"].min()} -> {initiation_ml["FiscalYear"].max()}')

print('\nOmission ML sample')
print(f'  rows: {len(omission_ml):,}')
print(f'  events: {omit_events:,}')
print(f'  non-events: {omit_non_events:,}')
print(f'  imbalance non-event/event: {omit_non_events / omit_events:.2f}:1' if omit_events else '  no omission events')
print(f'  year range: {omission_ml["FiscalYear"].min()} -> {omission_ml["FiscalYear"].max()}')


Target QA checks
------------------------------------------------------------
QA passed.

Initiation ML sample
  rows: 94,090
  events: 2,856
  non-events: 91,234
  imbalance non-event/event: 31.94:1
  year range: 1990 -> 2025

Omission ML sample
  rows: 72,638
  events: 2,928
  non-events: 69,710
  imbalance non-event/event: 23.81:1
  year range: 1990 -> 2025


## 10. QA Checks

Year-by-year event counts, missingness rates, and duplicate checks. These diagnostics are intentionally limited to data readiness; full descriptive statistics belong in the descriptive notebook.

In [11]:
print('Initiation events by year:')
print(initiation_ml.groupby('FiscalYear')['target_initiation'].sum().loc[lambda s: s.gt(0)].to_string())

print('\nOmission events by year:')
print(omission_ml.groupby('FiscalYear')['target_omission'].sum().loc[lambda s: s.gt(0)].to_string())

print('\nTop missing lagged features, initiation sample (%):')
print((initiation_ml[lagged_features].isna().mean() * 100).sort_values(ascending=False).head(20).round(2).to_string())

print('\nTop missing lagged features, omission sample (%):')
print((omission_ml[lagged_features].isna().mean() * 100).sort_values(ascending=False).head(20).round(2).to_string())

print('\nDuplicate checks:')
print('Initiation duplicate FirmID-FiscalYear:', int(initiation_ml.duplicated(['FirmID', 'FiscalYear']).sum()))
print('Omission duplicate FirmID-FiscalYear:', int(omission_ml.duplicated(['FirmID', 'FiscalYear']).sum()))

print('\nRows where all firm-level lagged features are missing:')
print('Initiation:', int(initiation_ml[firm_lagged_features].isna().all(axis=1).sum()))
print('Omission:', int(omission_ml[firm_lagged_features].isna().all(axis=1).sum()))

print('\nRows where all lagged features including DIVPREM are missing:')
print('Initiation:', int(initiation_ml[lagged_features].isna().all(axis=1).sum()))
print('Omission:', int(omission_ml[lagged_features].isna().all(axis=1).sum()))

Initiation events by year:
FiscalYear
1990     96
1991     75
1992     90
1993    143
1994    115
1995    213
1996    128
1997    104
1998     87
1999     78
2000     61
2001     59
2002     59
2003    125
2004    137
2005    127
2006     81
2007     78
2008     61
2009     41
2010     78
2011     74
2012    125
2013     85
2014     73
2015     56
2016     44
2017     44
2018     51
2019     40
2020     34
2021     54
2022     53
2023     34
2024     28
2025     25

Omission events by year:
FiscalYear
1990    107
1991    124
1992    109
1993    115
1994    123
1995     96
1996    107
1997    141
1998    123
1999    101
2000    108
2001    112
2002     91
2003     62
2004     34
2005     53
2006     81
2007     59
2008     91
2009    203
2010    117
2011     58
2012     43
2013     58
2014     49
2015     38
2016     66
2017     41
2018     48
2019     34
2020     62
2021    114
2022     37
2023     37
2024     42
2025     44

Top missing lagged features, initiation sample (%):
Interest

## 11. Export CSV Files

Write the full engineered panel (for descriptive merges) and two ML risk-set CSVs. Imputation, scaling, and class weighting are *not* applied here — they must be performed inside each CV fold in the modelling notebook to prevent information leakage.

In [13]:
# Export a compact full panel with current-year features and targets for audit/debugging.
full_panel_cols = common_id_cols + [
    'target_initiation', 'target_omission',
    'DividendPayer_DVC', 'weighted_mb_payers', 'weighted_mb_nonpayers',
] + all_features + lagged_features

full_panel = df.loc[:, [c for c in full_panel_cols if c in df.columns]].copy()

full_panel.to_csv(FULL_PANEL_CSV, index=False)
initiation_ml.to_csv(INITIATION_CSV, index=False)
omission_ml.to_csv(OMISSION_CSV, index=False)

print('Exported files:')
print(f'  Full engineered panel: {FULL_PANEL_CSV}')
print(f'  Initiation ML CSV:     {INITIATION_CSV}')
print(f'  Omission ML CSV:       {OMISSION_CSV}')
print(f'  Feature list CSV:      {FEATURES_CSV}')

Exported files:
  Full engineered panel: /Users/matyaslezatka/Documents/Thesis/Cleaned/data/dividend_full_engineered_panel_1963_2025.csv
  Initiation ML CSV:     /Users/matyaslezatka/Documents/Thesis/Cleaned/data/ml_dividend_initiations_1990_2025.csv
  Omission ML CSV:       /Users/matyaslezatka/Documents/Thesis/Cleaned/data/ml_dividend_omissions_1990_2025.csv
  Feature list CSV:      /Users/matyaslezatka/Documents/Thesis/Cleaned/data/ml_feature_list.csv


## 12. Modelling Notes

Key constraints for the downstream ML notebook:
- Use only `_lag1` columns as predictors.
- Perform imputation, scaling, and class weighting inside each training fold.
- `DIVPREM_lag1` is market-wide — same value for all firms in a given fiscal year.
- The omission control group consists of established payers with ≥ 2 consecutive annual payments who continue paying.

**Winsorization:** No winsorization is applied in this notebook. Features are exported at their raw lagged values; all outlier handling is delegated to the CV fold-level imputer in the modelling notebook.

The SAMPLE OVERVIEW output below may display two stale lines (`Winsorization: global 1st/99th pct` and `Exempt: DIVPREM_lag1 only`) that were carried over from a prior notebook execution and do not reflect any operation performed by the current code. Clear and re-run the cell to obtain clean output.

In [14]:
print('=' * 60)
print('SAMPLE OVERVIEW')
print('=' * 60)

print(f'\nUniverse: Compustat Fundamentals Annual, 1963-2025')
print(f'Estimation window: {START_ANALYSIS_YEAR}-{END_ANALYSIS_YEAR}')
print('Universe filters: Compustat INDL; TotalAssets > 0, CommonEquity > 0, Sales > 0; no market-cap screen')

print(f'\n{"-" * 60}')
print('INITIATION RISK SET')
print(f'{"-" * 60}')
print(f'  Firm-years:    {len(initiation_ml):>10,}')
print(f'  Events:        {int(initiation_ml["target_initiation"].sum()):>10,}')
print(f'  Non-events:    {int((initiation_ml["target_initiation"] == 0).sum()):>10,}')
print(f'  Imbalance:     {int((initiation_ml["target_initiation"] == 0).sum()) / int(initiation_ml["target_initiation"].sum()):>9.1f}:1')
print(f'  Event rate:    {initiation_ml["target_initiation"].mean() * 100:>9.2f}%')
print(f'  Year range:    {int(initiation_ml["FiscalYear"].min())}-{int(initiation_ml["FiscalYear"].max())}')
print(f'  Unique firms:  {initiation_ml["FirmID"].nunique():>10,}')

print(f'\n{"-" * 60}')
print('OMISSION RISK SET')
print(f'{"-" * 60}')
print(f'  Firm-years:    {len(omission_ml):>10,}')
print(f'  Events:        {int(omission_ml["target_omission"].sum()):>10,}')
print(f'  Non-events:    {int((omission_ml["target_omission"] == 0).sum()):>10,}')
print(f'  Imbalance:     {int((omission_ml["target_omission"] == 0).sum()) / int(omission_ml["target_omission"].sum()):>9.1f}:1')
print(f'  Event rate:    {omission_ml["target_omission"].mean() * 100:>9.2f}%')
print(f'  Year range:    {int(omission_ml["FiscalYear"].min())}-{int(omission_ml["FiscalYear"].max())}')
print(f'  Unique firms:  {omission_ml["FirmID"].nunique():>10,}')

print(f'\n{"-" * 60}')
print('PREDICTORS')
print(f'{"-" * 60}')
print(f'  Total features:  {len(lagged_features)}')
print(f'  Firm-level:      {len(firm_lagged_features)}')
print(f'  Market-wide:     {len(context_lagged_features)}')
print('=' * 60)

SAMPLE OVERVIEW

Universe: Compustat Fundamentals Annual, 1963-2025
Estimation window: 1990-2025
Universe filters: Compustat INDL; TotalAssets > 0, CommonEquity > 0, Sales > 0; no market-cap screen

------------------------------------------------------------
INITIATION RISK SET
------------------------------------------------------------
  Firm-years:        94,090
  Events:             2,856
  Non-events:        91,234
  Imbalance:          31.9:1
  Event rate:         3.04%
  Year range:    1990-2025
  Unique firms:      14,205

------------------------------------------------------------
OMISSION RISK SET
------------------------------------------------------------
  Firm-years:        72,638
  Events:             2,928
  Non-events:        69,710
  Imbalance:          23.8:1
  Event rate:         4.03%
  Year range:    1990-2025
  Unique firms:       6,964

------------------------------------------------------------
PREDICTORS
-----------------------------------------------------

In [15]:
# Diagnostic: what would sample counts be under DivPS-only dividend status?
# Uses the same raw data, accounting filter, listing-age rule, and risk-set logic.

tmp = raw.rename(columns=rename_dict).copy()

tmp['FiscalDate'] = pd.to_datetime(tmp['FiscalDate'], errors='coerce')
tmp['FiscalYear'] = pd.to_numeric(tmp['FiscalYear'], errors='coerce').astype('Int64')

id_cols_tmp = ['FirmID', 'Ticker', 'CUSIP', 'CompanyName', 'FIC', 'Currency']
for col in id_cols_tmp:
    if col in tmp.columns:
        tmp[col] = tmp[col].astype('string')

numeric_cols_tmp = [c for c in tmp.columns if c not in id_cols_tmp + ['FiscalDate']]
for col in numeric_cols_tmp:
    tmp[col] = pd.to_numeric(tmp[col], errors='coerce')

tmp = (
    tmp.sort_values(['FirmID', 'FiscalYear', 'FiscalDate'])
       .drop_duplicates(['FirmID', 'FiscalYear'], keep='last')
       .reset_index(drop=True)
)

tmp['first_compustat_year_full'] = tmp.groupby('FirmID')['FiscalYear'].transform('min')
tmp['listing_age'] = tmp['FiscalYear'] - tmp['first_compustat_year_full']

tmp['DivPS_Fiscal'] = pd.to_numeric(tmp['DivPS_Fiscal'], errors='coerce')
tmp['DivCash'] = pd.to_numeric(tmp['DivCash'], errors='coerce')
tmp['SharesOut'] = pd.to_numeric(tmp['SharesOut'], errors='coerce')

fallback_divps = tmp['DivCash'] / tmp['SharesOut'].replace(0, np.nan)
tmp['DivPS'] = (
    tmp['DivPS_Fiscal']
      .fillna(fallback_divps)
      .fillna(0)
      .clip(lower=0)
)

# Counterfactual payer definition: DivPS only
tmp['DividendPaid_alt'] = tmp['DivPS'].gt(0)
tmp['DividendPaidInt_alt'] = tmp['DividendPaid_alt'].astype('int8')

tmp['FiscalYear_lag'] = tmp.groupby('FirmID')['FiscalYear'].shift(1)
tmp['has_t_minus_1'] = tmp['FiscalYear'].sub(tmp['FiscalYear_lag']).eq(1).fillna(False)

tmp['FiscalYear_lag2'] = tmp.groupby('FirmID')['FiscalYear'].shift(2)
tmp['has_t_minus_2'] = tmp['FiscalYear_lag'].sub(tmp['FiscalYear_lag2']).eq(1).fillna(False)

tmp['DividendPaid_lag1_alt'] = (
    tmp.groupby('FirmID')['DividendPaid_alt']
       .shift(1)
       .astype('boolean')
       .fillna(False)
       .astype(bool)
)

tmp['DividendPaid_lag2_alt'] = (
    tmp.groupby('FirmID')['DividendPaid_alt']
       .shift(2)
       .astype('boolean')
       .fillna(False)
       .astype(bool)
)

tmp['PaidPriorEver_alt'] = (
    tmp.groupby('FirmID')['DividendPaidInt_alt']
       .transform(lambda s: s.shift(1).fillna(0).cummax())
       .astype(bool)
)

# Same accounting-base screen
tmp = tmp[
    tmp['TotalAssets'].gt(MIN_ASSETS) &
    tmp['CommonEquity'].gt(MIN_EQUITY) &
    tmp['Sales'].gt(0)
].copy().reset_index(drop=True)

tmp['target_initiation_alt'] = (
    tmp['DividendPaid_alt'] &
    (~tmp['PaidPriorEver_alt']) &
    tmp['has_t_minus_1'] &
    tmp['listing_age'].ge(MIN_MICHAELY_LISTING_AGE)
).astype('int8')

first_init_alt = (
    tmp.loc[tmp['target_initiation_alt'].eq(1)]
       .groupby('FirmID')['FiscalYear']
       .min()
       .rename('first_init_year_alt')
)

tmp = tmp.join(first_init_alt, on='FirmID')
tmp['target_initiation_alt'] = (
    tmp['target_initiation_alt'].eq(1) &
    tmp['FiscalYear'].eq(tmp['first_init_year_alt'])
).astype('int8')
tmp = tmp.drop(columns='first_init_year_alt')

tmp['target_omission_alt'] = (
    (~tmp['DividendPaid_alt']) &
    tmp['DividendPaid_lag1_alt'] &
    tmp['DividendPaid_lag2_alt'] &
    tmp['has_t_minus_1'] &
    tmp['has_t_minus_2']
).astype('int8')

analysis_year_mask_alt = tmp['FiscalYear'].between(START_ANALYSIS_YEAR, END_ANALYSIS_YEAR)

initiation_mask_alt = (
    analysis_year_mask_alt &
    (~tmp['PaidPriorEver_alt']) &
    ((~tmp['DividendPaid_alt']) | tmp['target_initiation_alt'].eq(1))
)

omission_mask_alt = (
    analysis_year_mask_alt &
    tmp['has_t_minus_1'] &
    tmp['has_t_minus_2'] &
    tmp['DividendPaid_lag1_alt'] &
    tmp['DividendPaid_lag2_alt']
)

init_alt = tmp.loc[initiation_mask_alt]
omit_alt = tmp.loc[omission_mask_alt]

print('=' * 60)
print('COUNTERFACTUAL SAMPLE USING DivPS ONLY')
print('=' * 60)
print(f'Initiation rows   : {len(init_alt):,}')
print(f'Initiation events : {int(init_alt["target_initiation_alt"].sum()):,}')
print(f'Initiation rate   : {100 * init_alt["target_initiation_alt"].mean():.2f}%')
print()
print(f'Omission rows     : {len(omit_alt):,}')
print(f'Omission events   : {int(omit_alt["target_omission_alt"].sum()):,}')
print(f'Omission rate     : {100 * omit_alt["target_omission_alt"].mean():.2f}%')
print('=' * 60)

COUNTERFACTUAL SAMPLE USING DivPS ONLY
Initiation rows   : 108,713
Initiation events : 2,738
Initiation rate   : 2.52%

Omission rows     : 66,320
Omission events   : 2,382
Omission rate     : 3.59%
